<a href="https://colab.research.google.com/github/guifclaro-cyber/fase1-pucrs-steam/blob/main/Fase_1_PPD_Steam_Guilherme_Farias_Claro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
#1 Import GDrive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
#2 Import CSV
import csv

caminho_csv = '/content/drive/MyDrive/fase1-ppd/steam_games.csv'

with open(caminho_csv, encoding='utf-8') as arquivo:
    leitor = csv.DictReader(arquivo)
    jogos = list(leitor)

print(f"Total de jogos carregados: {len(jogos)}")
print(jogos[0])

Total de jogos carregados: 72934
{'AppID': '20200', 'Name': 'Galactic Bowling', 'Release date': 'Oct 21, 2008', 'Estimated owners': '0 - 20000', 'Peak CCU': '0', 'Required age': '0', 'Price': '19.99', 'DLC count': '0', 'About the game': 'Galactic Bowling is an exaggerated and stylized bowling game with an intergalactic twist. Players will engage in fast-paced single and multi-player competition while being submerged in a unique new universe filled with over-the-top humor, wild characters, unique levels, and addictive game play. The title is aimed at players of all ages and skill sets. Through accessible and intuitive controls and game-play, Galactic Bowling allows you to jump right into the action. A single-player campaign and online play allow you to work your way up the ranks of the Galactic Bowling League! Whether you have hours to play or only a few minutes, Galactic Bowling is a fast paced and entertaining experience that will leave you wanting more! Full Single-player story campa

In [29]:
#3 CatalogoJogos, exceção e consultas
import csv
class ArquivoInvalidoError(Exception):
    """
    Exceção própria, sinalizada quando o arquivo CSV do catálogo
    não pode ser aberto.
    """
    pass

class CatalogoJogos:
    """
    Representa o catálogo de jogos da Steam carregado de um arquivo CSV.

    A classe esconde os detalhes de como o arquivo é lido e oferece
    métodos prontos para consultas sobre os jogos (porcentagem de
    jogos gratuitos, ano com mais lançamentos, etc).

    >>> catalogo = CatalogoJogos('steam_games.csv')
    >>> catalogo.total_de_jogos() > 0
    True
    """

    def __init__(self, caminho_arquivo):
        """
        Carrega o CSV e guarda os jogos internamente.

        caminho_arquivo: caminho para o arquivo .csv com os dados dos jogos.
        """
        self._jogos = self._carregar_csv(caminho_arquivo)

    def _carregar_csv(self, caminho_arquivo):
        """
        Método privado (com underline): lê o CSV e devolve uma
        lista de dicionários (um por jogo).

        Levanta ArquivoInvalidoError se o arquivo não puder ser aberto.
        """
        try:
            with open(caminho_arquivo, encoding='utf-8') as arquivo:
                leitor = csv.DictReader(arquivo)
                return list(leitor)
        except FileNotFoundError:
            raise ArquivoInvalidoError(
                f"Não foi possível encontrar o arquivo: {caminho_arquivo}"
            )

    def total_de_jogos(self):
        """Retorna quantos jogos existem no catálogo carregado."""
        return len(self._jogos)

    def porcentagem_gratuitos_pagos(self):
        """
        Calcula a porcentagem de jogos gratuitos e pagos no catálogo.

        Um jogo é gratuito quando o preço ('Price') é igual a 0.
        Retorna uma tupla (porcentagem_gratuitos, porcentagem_pagos),
        ambos arredondados em 2 casas decimais.

        >>> catalogo = CatalogoJogos('steam_games.csv')
        >>> gratuitos, pagos = catalogo.porcentagem_gratuitos_pagos()
        >>> round(gratuitos + pagos, 2)
        100.0

        Teste sobre a amostra de 20 jogos, conferido manualmente
        (4 jogos gratuitos de 20 = 20%):

        >>> catalogo_amostra = CatalogoJogos('amostra_20_jogos.csv')
        >>> catalogo_amostra.porcentagem_gratuitos_pagos()
        (20.0, 80.0)
        """
        total = self.total_de_jogos()
        qtd_gratuitos = 0

        for jogo in self._jogos:
            preco = float(jogo['Price'])
            if preco == 0:
                qtd_gratuitos += 1

        qtd_pagos = total - qtd_gratuitos
        porcentagem_gratuitos = round((qtd_gratuitos / total) * 100, 2)
        porcentagem_pagos = round((qtd_pagos / total) * 100, 2)
        return porcentagem_gratuitos, porcentagem_pagos

    def ano_com_mais_lancamentos(self):
        """
        Identifica os anos com o maior número de jogos novos lançados.
        O ano é extraído da coluna 'Release date' (ex: 'Oct 21, 2008' ou
        'May 2020') pegando sempre o último "pedaço" do texto, que é o
        ano com 4 dígitos. Retorna uma lista de anos, pois pode haver
        empate no maior número de lançamentos.

        >>> catalogo = CatalogoJogos('steam_games.csv')
        >>> catalogo.ano_com_mais_lancamentos()
        [2022]

        Teste sobre a amostra de 20 jogos, conferido manualmente
        (2022 aparece 5 vezes, o maior número na amostra):

        >>> catalogo_amostra = CatalogoJogos('amostra_20_jogos.csv')
        >>> catalogo_amostra.ano_com_mais_lancamentos()
        [2022]
        """
        contagem_por_ano = {}

        for jogo in self._jogos:
            data_texto = jogo['Release date'].strip()
            ano = int(data_texto.split()[-1])
            contagem_por_ano[ano] = contagem_por_ano.get(ano, 0) + 1

        maior_quantidade = max(contagem_por_ano.values())

        anos_com_maior_quantidade = [
            ano for ano, quantidade in contagem_por_ano.items()
            if quantidade == maior_quantidade
        ]

        return sorted(anos_com_maior_quantidade)

        # Pergunta 3
        #Dos em PT-BR, como se distribui a qt de jogos por faixa owners por categoria?
    def _minimo_da_faixa(self, faixa_texto):
        """
        Método privado: extrai o valor mínimo (o número antes do
        hífen) de uma faixa de 'Estimated owners', usado apenas para
        ordenar as faixas da maior para a menor.
        """
        minimo_texto = faixa_texto.split(' - ')[0]
        return int(minimo_texto)

    def contagem_por_faixa_owners(self, idioma, categoria):
        """
        Conta quantos jogos existem em cada faixa de 'Estimated owners',
        considerando apenas jogos que suportam um idioma específico (PT-BR)
        e pertencem a uma categoria específica (Single Player ou Multiplayer).

        Retorna uma lista de tuplas (faixa, quantidade), ordenada da
        faixa de maior número de owners para a menor.

        >>> catalogo_amostra = CatalogoJogos('amostra_20_jogos.csv')
        >>> catalogo_amostra.contagem_por_faixa_owners('Portuguese - Brazil', 'Single-player')
        [('0 - 20000', 1), ('0 - 0', 1)]
        >>> catalogo_amostra.contagem_por_faixa_owners('Portuguese - Brazil', 'Multi-player')
        []
        """
        contagem_por_faixa = {}
        for jogo in self._jogos:
            if idioma not in jogo['Supported languages']:
                continue
            if categoria not in jogo['Categories']:
                continue
            faixa = jogo['Estimated owners']
            contagem_por_faixa[faixa] = contagem_por_faixa.get(faixa, 0) + 1

        resultado = list(contagem_por_faixa.items())
        resultado.sort(key=lambda item: self._minimo_da_faixa(item[0]), reverse=True)
        return resultado

In [30]:
#4
import csv
import random

def gerar_amostra(caminho_csv_origem, caminho_csv_amostra, tamanho=20, semente=42):
    """
    Sorteia 'tamanho' jogos aleatórios do CSV de origem, excluindo os
    20 primeiros registros do arquivo original, e salva o resultado
    em um novo arquivo CSV.

    A 'semente' (seed) fixa o sorteio: rodar essa função de novo com a
    mesma semente sempre gera exatamente a mesma amostra. Isso é
    importante porque a amostra precisa ser criada uma única vez e
    reutilizada sempre a partir daí.
    """
    with open(caminho_csv_origem, encoding='utf-8') as arquivo:
        leitor = csv.DictReader(arquivo)
        todos_os_jogos = list(leitor)
        cabecalho = leitor.fieldnames

    jogos_elegiveis = todos_os_jogos[20:]

    random.seed(semente)
    jogos_sorteados = random.sample(jogos_elegiveis, tamanho)

    with open(caminho_csv_amostra, 'w', encoding='utf-8', newline='') as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=cabecalho)
        escritor.writeheader()
        escritor.writerows(jogos_sorteados)

    print(f"Amostra de {tamanho} jogos salva em: {caminho_csv_amostra}")

In [31]:
#5
gerar_amostra(
    caminho_csv_origem='/content/drive/MyDrive/fase1-ppd/steam_games.csv',
    caminho_csv_amostra='/content/drive/MyDrive/fase1-ppd/amostra_20_jogos.csv'
)

Amostra de 20 jogos salva em: /content/drive/MyDrive/fase1-ppd/amostra_20_jogos.csv


In [32]:
%cd /content/drive/MyDrive/fase1-ppd

/content/drive/MyDrive/fase1-ppd


In [33]:
#6 Teste Exceção
try:
    catalogo_invalido = CatalogoJogos('arquivo_que_nao_existe.csv')
except ArquivoInvalidoError as erro:
    print(f"Erro esperado capturado: {erro}")

Erro esperado capturado: Não foi possível encontrar o arquivo: arquivo_que_nao_existe.csv


In [34]:
#Resposta Pergunta 3
catalogo = CatalogoJogos('/content/drive/MyDrive/fase1-ppd/steam_games.csv')
print("Single-player:")
for faixa, qtd in catalogo.contagem_por_faixa_owners('Portuguese - Brazil', 'Single-player'):
    print(f"  {faixa}: {qtd} jogos")

print("Multi-player:")
for faixa, qtd in catalogo.contagem_por_faixa_owners('Portuguese - Brazil', 'Multi-player'):
    print(f"  {faixa}: {qtd} jogos")

Single-player:
  20000000 - 50000000: 12 jogos
  10000000 - 20000000: 10 jogos
  5000000 - 10000000: 35 jogos
  2000000 - 5000000: 141 jogos
  1000000 - 2000000: 187 jogos
  500000 - 1000000: 265 jogos
  200000 - 500000: 484 jogos
  100000 - 200000: 459 jogos
  50000 - 100000: 551 jogos
  20000 - 50000: 837 jogos
  0 - 20000: 4158 jogos
  0 - 0: 447 jogos
Multi-player:
  100000000 - 200000000: 1 jogos
  50000000 - 100000000: 4 jogos
  20000000 - 50000000: 16 jogos
  10000000 - 20000000: 21 jogos
  5000000 - 10000000: 28 jogos
  2000000 - 5000000: 103 jogos
  1000000 - 2000000: 125 jogos
  500000 - 1000000: 173 jogos
  200000 - 500000: 202 jogos
  100000 - 200000: 172 jogos
  50000 - 100000: 173 jogos
  20000 - 50000: 247 jogos
  0 - 20000: 766 jogos
  0 - 0: 120 jogos


In [35]:
import doctest
doctest.testmod(verbose=True)

Trying:
    catalogo = CatalogoJogos('steam_games.csv')
Expecting nothing
ok
Trying:
    catalogo.total_de_jogos() > 0
Expecting:
    True
ok
Trying:
    catalogo = CatalogoJogos('steam_games.csv')
Expecting nothing
ok
Trying:
    catalogo.ano_com_mais_lancamentos()
Expecting:
    [2022]
ok
Trying:
    catalogo_amostra = CatalogoJogos('amostra_20_jogos.csv')
Expecting nothing
ok
Trying:
    catalogo_amostra.ano_com_mais_lancamentos()
Expecting:
    [2022]
ok
Trying:
    catalogo_amostra = CatalogoJogos('amostra_20_jogos.csv')
Expecting nothing
ok
Trying:
    catalogo_amostra.contagem_por_faixa_owners('Portuguese - Brazil', 'Single-player')
Expecting:
    [('0 - 20000', 1), ('0 - 0', 1)]
ok
Trying:
    catalogo_amostra.contagem_por_faixa_owners('Portuguese - Brazil', 'Multi-player')
Expecting:
    []
ok
Trying:
    catalogo = CatalogoJogos('steam_games.csv')
Expecting nothing
ok
Trying:
    gratuitos, pagos = catalogo.porcentagem_gratuitos_pagos()
Expecting nothing
ok
Trying:
    round(

TestResults(failed=0, attempted=14)